## Setup

In [ ]:
dbutils.library.restartPython()

In [ ]:
!pip install "httpcore[asyncio]"
!pip install --upgrade pip
!pip install requests
!pip install pandas
!pip install openpyxl
!pip install --upgrade httpx
!pip install httpx
!pip install nest_asyncio
!pip install asyncio
!pip install python-dotenv

dbutils.library.restartPython()

In [ ]:
import requests
import os
import pandas as pd
import openpyxl
import json
from pyspark.sql.functions import explode, col
import nest_asyncio
import asyncio
import httpx
import time
from datetime import datetime

from config import get_config
from config.secrets import get_campo_api_token

In [ ]:
# ------- SUMARIO DE EXECUCAO -------
execucao_steps = []

def log_step(etapa, df=None, status="Sucesso", observacoes="", registros_lidos=None, registros_escritos=None):
    n = df.count() if df is not None else 0
    execucao_steps.append({
        "etapa": etapa,
        "status": status,
        "registros_lidos": registros_lidos if registros_lidos is not None else n,
        "registros_escritos": registros_escritos if registros_escritos is not None else n,
        "flags": _collect_flags(df) if df is not None else "\u2014",
        "observacoes": observacoes,
    })

def _collect_flags(df, colunas=None):
    from pyspark.sql import functions as F
    flag_cols = colunas if colunas is not None else [c for c in df.columns if c.startswith("flag_")]
    triggered = []
    for col_name in flag_cols:
        count = df.filter(F.col(col_name).isNotNull() & (F.col(col_name) != "")).count()
        if count > 0:
            triggered.append(f"{col_name}({count})")
    return "; ".join(triggered) if triggered else "\u2014"

def persistir_log():
    from pyspark.sql import Row
    import datetime as _dt
    try:
        notebook_name = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        _execution_log_path = project_path + "/execution_log"
        _batch_ts = _dt.datetime.now().isoformat()

        _log_rows = [
            Row(
                batch_ts=_batch_ts,
                projeto=projeto,
                notebook=notebook_name.split("/")[-1],
                ordem=i,
                etapa=s["etapa"],
                status=s["status"],
                registros_lidos=int(s.get("registros_lidos") or 0),
                registros_escritos=int(s.get("registros_escritos") or 0),
                flags=str(s.get("flags") or "\u2014"),
                observacoes=str(s.get("observacoes") or ""),
                ts_registro=_dt.datetime.now().isoformat(),
            )
            for i, s in enumerate(execucao_steps)
        ]
        _log_df = spark.createDataFrame(_log_rows)
        _log_df.write.format("delta").mode("append").option("mergeSchema", "true").save(_execution_log_path)
        print(f"Log persistido: {len(_log_rows)} steps -> {_execution_log_path}")
    except Exception as _log_err:
        print(f"Aviso: falha ao persistir log de execucao \u2014 {_log_err}")

## Projeto e periodo

Widget `projeto` seleciona a campanha (chave de `config/projetos.py`, ex.: `1233_IC_BA` ou `1233_IC_CDM`). Todo caminho, filtro e destino do restante do notebook vem de `cfg`, nunca de uma string fixa — assim uma execucao nunca mistura dados/arquivos de outro projeto.

In [ ]:
from datetime import datetime, timezone, timedelta

# Widgets (job parameters)
dbutils.widgets.text("projeto", "")
dbutils.widgets.text("inicio", "")
dbutils.widgets.text("fim", "")

projeto = dbutils.widgets.get("projeto").strip()
inicio = dbutils.widgets.get("inicio").strip()
fim    = dbutils.widgets.get("fim").strip()

cfg = get_config(projeto)  # falha alto se `projeto` estiver vazio ou nao cadastrado

def iso_z(dt: datetime) -> str:
    return dt.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"

def last_sunday_00_utc(ref_dt_utc: datetime) -> datetime:
    # Monday=0 ... Sunday=6
    days_since_sunday = (ref_dt_utc.weekday() + 1) % 7
    d = (ref_dt_utc.date() - timedelta(days=days_since_sunday))
    return datetime(d.year, d.month, d.day, 0, 0, 0, tzinfo=timezone.utc)

# Se o job nao passou os valores, calcula:
# inicio = penultimo domingo 00:00:00.000Z
# fim    = ultimo domingo 00:00:00.000Z
if not inicio or not fim:
    now_utc = datetime.now(timezone.utc)
    fim_dt = last_sunday_00_utc(now_utc)
    inicio_dt = fim_dt - timedelta(days=7)

    inicio = inicio or iso_z(inicio_dt)
    fim    = fim    or iso_z(fim_dt)

In [ ]:
project_name = projeto
project_path = "/mnt/wst/" + project_name
# Define path
folder_bronze = project_path + "/BRONZE/"
api_path = folder_bronze + "api/"

# dbutils.fs.mkdirs(folder_bronze)
# dbutils.fs.mkdirs(api_path)

assert dbutils.fs.ls(project_path), f"Project path {project_path} is invalid."

### Table Extraction from `API`

> ##### Replace 'params' with the correct time period (YYYY-MM-DDT00:00:00.000Z)

In [ ]:
params = {
        "inicio": inicio,
        "fim": fim
    }

In [ ]:
from datetime import datetime, timedelta
import asyncio
import httpx
import pandas as pd
import nest_asyncio

nest_asyncio.apply()

BASE_URL = "https://campoanalises.com.br/wst/amostras"  # mesma API/token para todos os projetos
token = get_campo_api_token()
headers = {"Authorization": f"Bearer {token}"}
timeout_config = httpx.Timeout(connect=30.0, read=30.0, write=30.0, pool=30.0)

log_info = []
MAX_DAYS = 60

def to_iso_z(dt: datetime) -> str:
    return dt.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"

async def fetch_range(client, inicio_range, fim_range, max_retries=3):
    params = {
        "inicio": inicio_range.strftime("%Y-%m-%d"),
        "fim": fim_range.strftime("%Y-%m-%d")
    }
    for attempt in range(1, max_retries + 1):
        try:
            response = await client.get(BASE_URL, headers=headers, params=params)
            response.raise_for_status()
            result_json = response.json()
            amostras = result_json.get("amostrasHga", [])
            if not amostras:
                log_info.append({
                    "status": "Sem amostras",
                    "n_amostras": 0,
                    "inicio": params["inicio"],
                    "fim": params["fim"],
                    "error_msg": ""
                })
                return []
            log_info.append({
                "status": "Sucesso",
                "n_amostras": len(amostras),
                "inicio": params["inicio"],
                "fim": params["fim"],
                "error_msg": ""
            })
            return amostras
        except Exception as exc:
            if attempt == max_retries:
                log_info.append({
                    "status": f"Erro: {type(exc).__name__}",
                    "n_amostras": None,
                    "inicio": params["inicio"],
                    "fim": params["fim"],
                    "error_msg": str(exc)
                })
                return []
            await asyncio.sleep(attempt * 2)

async def fetch(client):
    inicio_dt = datetime.strptime(inicio, "%Y-%m-%dT%H:%M:%S.%fZ")
    fim_dt = datetime.strptime(fim, "%Y-%m-%dT%H:%M:%S.%fZ")
    all_amostras = []
    current_start = inicio_dt
    while current_start <= fim_dt:
        current_end = min(current_start + timedelta(days=MAX_DAYS - 1), fim_dt)
        amostras = await fetch_range(client, current_start, current_end)
        all_amostras.extend(amostras)
        current_start = current_end + timedelta(days=1)
    if all_amostras:
        return {"amostrasHga": all_amostras}
    return None

async def main():
    async with httpx.AsyncClient(timeout=timeout_config) as client:
        result = await fetch(client)
        return result

json_data = await main()

df_log = pd.DataFrame(log_info)
df_amostras = pd.json_normalize(json_data["amostrasHga"]) if json_data else pd.DataFrame()

In [ ]:
df_log.display()

In [ ]:
blocos_com_erro = [row for row in log_info if row["status"].startswith("Erro")]
blocos_sem_amostras = [row for row in log_info if row["status"] == "Sem amostras"]
total_amostras = sum(row["n_amostras"] or 0 for row in log_info)

detalhe_erros = [
    f'{row["inicio"]} a {row["fim"]} ({row["status"]}: {row["error_msg"]})'
    for row in blocos_com_erro
]

if blocos_com_erro and total_amostras == 0:
    status_etapa = "Erro"
elif blocos_com_erro:
    status_etapa = "Aviso"
else:
    status_etapa = "Sucesso"

execucao_steps.append({
    "etapa": "Requisicao na API",
    "status": status_etapa,
    "registros_lidos": total_amostras,
    "registros_escritos": total_amostras,
    "flags": f"blocos_com_erro({len(blocos_com_erro)}); blocos_sem_amostras({len(blocos_sem_amostras)})" if (blocos_com_erro or blocos_sem_amostras) else "\u2014",
    "observacoes": (
        f"{len(log_info)} blocos de datas consultados, {total_amostras} amostras recebidas"
        if not blocos_com_erro
        else f"Falha em {len(blocos_com_erro)} bloco(s): {'; '.join(detalhe_erros)}"
    )
})

##### JSON to df_spark

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import lit

def parse_hga_api_json_to_df(api_data, origem_arquivo=None, pages_id=None):
    amostras = api_data.get("amostrasHga", [])
    linhas = []

    for amostra in amostras:
        parametros = amostra.get("parametros", [])
        if parametros[0].get("idAmostra") in []:
            print(json.dumps(amostra, indent=2, ensure_ascii=False))

        cadeia_cust = amostra.get("cadeiaDeCustodia")
        cadeia_cust_info = cadeia_cust.get("informacoes")
        cadeia_cust_local = cadeia_cust.get("avaliacoesDoLocal", [])

        avaliacoes_local = {}

        for item in cadeia_cust_local:
            label = item.get("label", "")

            # normaliza label para virar nome de coluna
            label_norm = (
                label.strip()
                .replace(":", "")
                .replace(" ", "_")
                .replace("/", "_")
                .replace("-", "_")
            )

            values = item.get("value", [])

            # compila informacoes da subsessao de avaliacoes do local
            avaliacoes_local[f"CC_{label_norm}"] = " | ".join(
                [str(v) for v in values if v is not None and str(v).strip() != ""]
            )

        for parametro in parametros:
            linha = {
                # Dados da amostra
                "codigoHga": amostra.get("codigoHga"),
                "idAmostra": amostra.get("idAmostra"),
                "nomeAmostra": amostra.get("nomeAmostra"),
                "descricaoAmostra": amostra.get("descricaoAmostra"),
                "matriz": amostra.get("matriz"),
                "dataHoraAmostragem": amostra.get("DataHoraAmostragem"),
                "codigoQualidade": amostra.get("codigoQualidade"),
                "frequencia": amostra.get("frequencia"),
                "laboratorio": amostra.get("laboratorio"),
                "comentario_amostra": amostra.get("comentario"),
                "parent": amostra.get("parent"),
                "dataEnvioLab": amostra.get("dataEnvioLab"),
                "dataRecebLab": amostra.get("dataRecebLab"),
                "profInicial": amostra.get("profInicial"),
                "profFinal": amostra.get("profFinal"),
                "composta": str(amostra.get("composta")),
                "coletada": str(amostra.get("coletada")),
                "acreditacao": amostra.get("acreditacao"),
                "legislacao": amostra.get("legislacao"),
                "finalidade": amostra.get("finalidade"),
                "campanha": amostra.get("campanha"),
                "motivoNaoColeta": amostra.get("motivoNaoColeta"),
                "dataLiberacao": amostra.get("dataLiberacao"),

                # Dados do parametro
                "parametroPadrao": parametro.get("parametroPadrao"),
                "resultadoNumerico": parametro.get("resultadoNumerico"),
                "dataHoraAnalise": parametro.get("dataHoraAnalise"),
                "resultadoTexto": parametro.get("resultadoTexto"),
                "unidadePadrao": parametro.get("unidadePadrao"),
                "qualifier": parametro.get("qualifier"),
                "limiteDeteccao": parametro.get("limiteDeteccao"),
                "limiteQuantificacao": parametro.get("limiteQuantificacao"),
                "parametroOriginal": parametro.get("parametroOriginal"),
                "resultadoOriginal": parametro.get("resultadoOriginal"),
                "unidadeOriginal": parametro.get("unidadeOriginal"),
                "metodoAnalise": parametro.get("metodoAnalise"),
                "comentario_parametro": parametro.get("comentario"),
                "tipoAnalise": parametro.get("tipoAnalise"),
                "reportavel": str(parametro.get("reportavel")),
                "qaqcFlag": str(parametro.get("qaqcFlag")),
                "acreditacaoMetodo": parametro.get("acreditacaoMetodo"),

                # Dados da cadeia de custodia
                "CC_tipoAmostra": cadeia_cust_info.get("tipoAmostra"),
                "CC_coordenadas": cadeia_cust_info.get("coordenadas"),
                "CC_dataColeta": cadeia_cust_info.get("dataColeta"),
                "CC_metodoAmostragem": cadeia_cust_info.get("metodoAmostragem"),
                "CC_dataRecebimentoAmostras": cadeia_cust_info.get("dataRecebimentoAmostras"),
                "CC_temperaturaRecebimentoAmostra": cadeia_cust_info.get("temperaturaRecebimentoAmostra"),
                "CC_avaliacaoIntegridadeAmostra": cadeia_cust_info.get("avaliacaoIntegridadeAmostra"),
                "CC_planoDeAmostra": cadeia_cust_info.get("planoDeAmostra"),
                "CC_observacoes": cadeia_cust_info.get("observacoes"),
                "CC_responsavelPelaColeta": cadeia_cust_info.get("responsavelPelaColeta"),

                # Dados da avaliacao do local (cadeia cust tambem)
                **avaliacoes_local

            }
            linhas.append(linha)

    if not linhas:
        raise ValueError("Nenhum dado encontrado no JSON.")

    # Criacao dinamica do schema
    all_keys = list(linhas[0].keys())
    schema = StructType([StructField(k, StringType(), True) for k in all_keys])

    # Criar DataFrame
    df = spark.createDataFrame(linhas, schema=schema)

    # Adicionar colunas extras, se existirem
    if origem_arquivo:
        df = df.withColumn("OrigemArquivo", lit(origem_arquivo))
    if pages_id:
        df = df.withColumn("pages_id", lit(pages_id))

    return df


In [ ]:
dfs = []
sem_dados = False

try:
    if json_data:
        df = parse_hga_api_json_to_df(json_data, origem_arquivo="API_CAMPO")
        dfs.append(df)

        df_final = dfs[0]
        for df in dfs[1:]:
            df_final = df_final.unionByName(df)

        execucao_steps.append({
            "etapa": "Parse JSON para DataFrame Spark",
            "status": "Sucesso",
            "registros_lidos": len(json_data.get("amostrasHga", [])),
            "registros_escritos": df_final.count(),
            "flags": "\u2014",
            "observacoes": "JSON convertido em DataFrame Spark com sucesso"
        })
    else:
        execucao_steps.append({
            "etapa": "Parse JSON para DataFrame Spark",
            "status": "Aviso",
            "registros_lidos": 0,
            "registros_escritos": 0,
            "flags": "\u2014",
            "observacoes": "Nenhum dado retornado da API nesta execucao \u2014 encerrando sem gravar na Bronze"
        })
        sem_dados = True

except Exception as _e:
    execucao_steps.append({
        "etapa": "Parse JSON para DataFrame Spark",
        "status": "Erro",
        "registros_lidos": 0,
        "registros_escritos": 0,
        "flags": "\u2014",
        "observacoes": str(_e)
    })
    persistir_log()
    raise

if sem_dados:
    persistir_log()
    dbutils.notebook.exit("Sem dados retornados da API")

In [ ]:
if 'df_final' in globals():
    registros_antes_filtro = df_final.count()
    df_final = df_final.filter(col('campanha').isin(cfg["campanha_api"]))
    registros_depois_filtro = df_final.count()

    execucao_steps.append({
        "etapa": "Filtro de campanha",
        "status": "Sucesso" if registros_depois_filtro > 0 else "Aviso",
        "registros_lidos": registros_antes_filtro,
        "registros_escritos": registros_depois_filtro,
        "flags": "\u2014",
        "observacoes": (
            f"{registros_antes_filtro - registros_depois_filtro} registros descartados (campanha diferente de '{cfg['campanha_api']}')"
            if registros_depois_filtro < registros_antes_filtro
            else "Todos os registros pertencem a campanha esperada"
        )
    })
else:
    print("df_final nao existe \u2014 etapa de Filtro de campanha ignorada (sem dados da API)")

##### The Delta table path for saving API data follows this structure:
**_'apiCAMPO (StartDate) (EndDate) _ (SaveDateTime)'_**


#### Components:
- **`<StartDate>`**: Start date from API parameters, formatted as `yyyyMMdd`
- **`<EndDate>`**: End date from API parameters, formatted as `yyyyMMdd`
- **`<SaveDateTime>`**: Current timestamp at save time, formatted as `yyyyMMdd_HHmm`

#### Example:
apiCAMPO_20220113_20220130_20240808_1530


In [ ]:
if 'df_final' in globals():
    try:
        # Extrair as datas compactadas dos parametros
        inicio_fmt = params['inicio'][:10].replace("-", "")
        fim_fmt = params['fim'][:10].replace("-", "")

        # Data e hora atual
        agora_fmt = datetime.now().strftime("%Y%m%d_%H%M")

        # Nome do diretorio de salvamento
        nome_pasta = f"apiCAMPO_{inicio_fmt}_{fim_fmt}_{agora_fmt}"
        caminho_completo = f"{api_path}/{nome_pasta}"

        # Salvar o DataFrame em Delta
        df_final.write.format("delta").mode("overwrite").save(caminho_completo)
        print(f"Nome da pasta de salvamento: {nome_pasta}")

        execucao_steps.append({
            "etapa": "Gravacao Bronze",
            "status": "Sucesso",
            "registros_lidos": df_final.count(),
            "registros_escritos": df_final.count(),
            "flags": "\u2014",
            "observacoes": f"Salvo em {caminho_completo}"
        })
    except Exception as _e:
        execucao_steps.append({
            "etapa": "Gravacao Bronze",
            "status": "Erro",
            "registros_lidos": df_final.count(),
            "registros_escritos": 0,
            "flags": "\u2014",
            "observacoes": str(_e)
        })
        persistir_log()
        raise
else:
    print("df_final nao existe \u2014 etapa de Gravacao Bronze ignorada (sem dados da API)")

In [ ]:
display(pd.DataFrame(execucao_steps))

In [ ]:
persistir_log()

In [ ]:
dbutils.jobs.taskValues.set(
    key="output_filename",
    value=nome_pasta
)